# Download SOOP Dataset (OpenNeuro ds004889)

This notebook downloads the SOOP stroke dataset and packs it into a **tar archive** (Kaggle limits output to 500 files).

**After running:**
1. Click "Save Version" -> "Save & Run All (Commit)"
2. Wait for completion
3. In training notebook: Add Input -> Your Work -> this notebook's output

**Requirements:**
- Internet: ON
- Accelerator: None (GPU not needed)
- Runtime: ~30-40 min

In [ ]:
!pip install -q awscli

In [ ]:
import os
from pathlib import Path

# Download to /tmp to avoid filling /kaggle/working before tar
DOWNLOAD_DIR = Path("/tmp/soop/ds004889")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

!df -h /kaggle/working
!df -h /tmp

In [ ]:
# Step 1: Download participants.tsv
print("Downloading participants.tsv...")
!aws s3 cp --no-sign-request \
    s3://openneuro.org/ds004889/participants.tsv \
    {DOWNLOAD_DIR}/participants.tsv

# Check it
import csv
with open(DOWNLOAD_DIR / "participants.tsv") as f:
    reader = csv.DictReader(f, delimiter="\t")
    rows = list(reader)
print(f"Total participants: {len(rows)}")
print(f"Columns: {list(rows[0].keys())}")

# Show unique values for stroke column to find correct name
for col in rows[0].keys():
    if "stroke" in col.lower() or "isch" in col.lower():
        vals = set(r.get(col, "") for r in rows)
        print(f"  Column '{col}': unique values = {vals}")

In [ ]:
# Step 2: Download DWI + ADC (.nii.gz only)
print("Downloading DWI + ADC files...")
print("(This takes ~5-10 min)")
!aws s3 sync --no-sign-request \
    s3://openneuro.org/ds004889/ {DOWNLOAD_DIR}/ \
    --exclude "*" \
    --include "sub-*/dwi/*.nii.gz" \
    --no-progress

print("\nDWI+ADC download complete!")
!du -sh {DOWNLOAD_DIR}

In [ ]:
# Step 3: Download FLAIR only (skip T1w)
print("Downloading FLAIR files...")
print("(This takes ~5-10 min)")
!aws s3 sync --no-sign-request \
    s3://openneuro.org/ds004889/ {DOWNLOAD_DIR}/ \
    --exclude "*" \
    --include "sub-*/anat/*FLAIR*.nii.gz" \
    --no-progress

print("\nFLAIR download complete!")
!du -sh {DOWNLOAD_DIR}

In [ ]:
# Step 4: Download lesion masks
print("Downloading lesion masks...")
!aws s3 sync --no-sign-request \
    s3://openneuro.org/ds004889/derivatives/ {DOWNLOAD_DIR}/derivatives/ \
    --exclude "*" \
    --include "lesion_masks/*" \
    --include "lesion_masks/**/*" \
    --no-progress

print("\nMasks download complete!")
!du -sh {DOWNLOAD_DIR}
!du -sh {DOWNLOAD_DIR}/derivatives/ 2>/dev/null || echo "No derivatives dir"

In [ ]:
# Step 5: Verify download
soop_subs = sorted([d.name for d in DOWNLOAD_DIR.iterdir() if d.name.startswith("sub-")])
print(f"Total subject folders: {len(soop_subs)}")

dwi_count = adc_count = flair_count = mask_count = 0
for sub_id in soop_subs:
    sub_dir = DOWNLOAD_DIR / sub_id
    dwi_dir = sub_dir / "dwi"
    anat_dir = sub_dir / "anat"
    if dwi_dir.exists():
        for f in dwi_dir.iterdir():
            if "TRACE" in f.name: dwi_count += 1
            if "ADC" in f.name: adc_count += 1
    if anat_dir.exists():
        for f in anat_dir.iterdir():
            if "FLAIR" in f.name: flair_count += 1

mask_dir = DOWNLOAD_DIR / "derivatives" / "lesion_masks"
if mask_dir.exists():
    mask_count = len(list(mask_dir.rglob("*.nii.gz")))

print(f"\nFile counts:")
print(f"  DWI (TRACE): {dwi_count}")
print(f"  ADC:         {adc_count}")
print(f"  FLAIR:       {flair_count}")
print(f"  Masks:       {mask_count}")
print(f"\nSample subject: {soop_subs[0] if soop_subs else 'none'}")
if soop_subs:
    for root, dirs, files in os.walk(DOWNLOAD_DIR / soop_subs[0]):
        for f in files:
            print(f"  {os.path.relpath(os.path.join(root, f), DOWNLOAD_DIR / soop_subs[0])}")

In [ ]:
# Step 6: Pack into tar archive (Kaggle limits output to 500 files!)
import time

print("Packing SOOP into tar archive...")
print("(This takes ~5 min)")
t0 = time.time()

# tar without compression (files are already .nii.gz)
!cd /tmp/soop && tar cf /kaggle/working/soop_ds004889.tar ds004889/

elapsed = time.time() - t0
tar_size = os.path.getsize("/kaggle/working/soop_ds004889.tar") / 1e9
print(f"\nDone in {elapsed/60:.1f} min")
print(f"Archive: /kaggle/working/soop_ds004889.tar ({tar_size:.1f} GB)")

# Clean up download dir to free space
!rm -rf /tmp/soop

!df -h /kaggle/working
print("\n" + "="*60)
print("SOOP download & archive complete!")
print("="*60)
print("\nOutput: soop_ds004889.tar (1 file)")
print('Save this notebook, then add its output as Input to training notebook.')